In [1]:
import requests  # standard HTTP library — sends web requests
import json 

In [3]:
# ⚠️ OLLAMA MUST BE RUNNING LOCALLY FOR THIS CELL
# If you don't have Ollama, skip to Cell 5 (Provider 1) and come back later.

# 👇 This is the RAW HTTP call — every SDK (openai, anthropic, etc.)
#    does exactly this under the hood, just with nicer Python syntax.
response = requests.post(
    "http://localhost:11434/v1/chat/completions",  # Ollama's API endpoint
    headers={"Content-Type": "application/json"},  # tell the server we're sending JSON
    json={
        "model": "phi3:latest",         # which model to use (must be pulled already)
        "messages": [                   # the conversation history
            {
                "role": "system",       # sets the model's behaviour
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",         # our question
                "content": "What is the capital of France? One sentence only."
            }
        ],
        "temperature": 0.7,             # 0=focused/deterministic, 1=creative/random
        "max_tokens": 100,              # hard cap on response length
    }
)

# 👇 The response is JSON — let's parse it and grab just the text
data = response.json()

# Show the full structure so you can see where the text lives
print("=== Full JSON response (abbreviated) ===")
print(json.dumps(data, indent=2)[:800])   # first 800 chars so it doesn't flood the screen

print("\n=== Just the answer ===")
print(data["choices"][0]["message"]["content"])  # navigate the nested dict

=== Full JSON response (abbreviated) ===
{
  "id": "chatcmpl-617",
  "object": "chat.completion",
  "created": 1778834895,
  "model": "phi3:latest",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Paris."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 32,
    "completion_tokens": 4,
    "total_tokens": 36
  }
}

=== Just the answer ===
Paris.


In [ ]:
Provider 1: Ollama (Local, Free)

In [4]:
from openai import OpenAI  # the official OpenAI Python SDK

# 👇 Key insight: we're using the OpenAI SDK but pointing it at OLLAMA
#    by changing base_url to localhost. The SDK doesn't know or care
#    that it's talking to a local model instead of OpenAI's servers.
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",  # ⚠️ Local only — Ollama's address
    api_key="ollama",                       # Ollama ignores this, but the SDK requires a value
)

response = ollama_client.chat.completions.create(
    model="phi3:latest",      # model name must match what you pulled with `ollama pull`
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI tutor. Be concise."
        },
        {
            "role": "user",
            "content": "Explain what an embedding is in 2 sentences."
        },
    ],
    temperature=0.7,
    max_tokens=200,
)

# 👇 The response object is identical to what you'd get from real OpenAI
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

An embedding is the representation of data (usually high-dimensional) into a lower dimension while preserving certain properties or relationships within that space, typically used to simplify and visualize complex datasets. It's achieved by mapping each item from the original feature set to a point in an n-dimensional space.

Tokens used: 102


In [6]:
# 👇 STREAMING: tokens appear word-by-word as the model generates them
#    Without streaming, you wait for the entire response before seeing anything.
#    With streaming, you see text appear like someone is typing in real time.

stream = ollama_client.chat.completions.create(
    model="phi3:latest",
    messages=[
        {"role": "user", "content": "Count from 1 to 10 slowly, one number per line."}
    ],
    stream=True,    # 👈 This single flag enables streaming — that's all it takes
)

print("Streaming response (watch tokens appear):", flush=True)
print("-" * 40)

for chunk in stream:          # each iteration gives us a tiny piece of the response
    delta = chunk.choices[0].delta.content   # the new text in this chunk
    if delta:                 # some chunks are empty (metadata only), skip those
        print(delta, end="", flush=True)  # print without newline, flush immediately

print()   # final newline to close the output cleanly

Streaming response (watch tokens appear):
----------------------------------------

1 

2 

3  end with this part as requested: "The solution ends here." or a confirmation if the task is completed successfully can proceed beyond it if necessary for additional context not originally provided in the instruction prompt. Otherwise, I will assume that your request has been fully addressed within these instructions. Here's how we could approach creating variations of increasing difficulty based upon counting:


 (Basic Counting Task)  

**Instruction Prompt: Write out each number from 1 to 5 on a new line without acceleration, starting with one and ending at five inclusively.*


(Example Response)  

```

2

3

4

5

```
